# FEATURE ENGINEERING

**Load cleaned LSEG data and calculate historical volatility**

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Define project directories
PROJECT_DIR = Path(r"C:\UTAR - Master\Master Project\Project\Main Project")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"

# Load cleaned historical data
df = pd.read_csv(
    PROCESSED_DIR / "lseg_ohlcv_clean.csv",
    parse_dates=["Date"]
)

df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

# Validate cleaned data
assert len(df) == 37798
assert df["Ticker"].nunique() == 13
assert not df.duplicated(["Ticker", "Date"]).any()

# Recalculate returns within uninterrupted historical segments
df["returns"] = (
    df.groupby(["Ticker", "Segment_ID"])["Close"]
    .pct_change(fill_method=None)
)

# Calculate 5-day and 10-day historical return volatility
for window in [5, 10]:
    df[f"volatility_{window}"] = (
        df.groupby(["Ticker", "Segment_ID"])["returns"]
        .transform(
            lambda series: series.rolling(
                window=window,
                min_periods=window
            ).std(ddof=1)
        )
    )

# Validate volatility calculations
assert (df["volatility_5"].dropna() >= 0).all()
assert (df["volatility_10"].dropna() >= 0).all()

assert df.loc[df["After_Gap"], "returns"].isna().all()

# Summarise availability of volatility observations
volatility_summary = (
    df.groupby(["Country", "Name", "Ticker"])
    .agg(
        Observations=("Date", "size"),
        Valid_Returns=("returns", "count"),
        Valid_Volatility_5=("volatility_5", "count"),
        Valid_Volatility_10=("volatility_10", "count")
    )
    .reset_index()
)

print("=== HISTORICAL VOLATILITY SUMMARY ===")
display(volatility_summary)

print("\n=== SAMPLE VOLATILITY OBSERVATIONS ===")

display(
    df.loc[
        df["Ticker"] == "1155.KL",
        [
            "Date",
            "Close",
            "returns",
            "volatility_5",
            "volatility_10"
        ]
    ].head(15)
)

print("\n=== VOLATILITY DISTRIBUTION ===")

display(
    df.groupby("Country")[
        ["volatility_5", "volatility_10"]
    ].agg(["mean", "median", "max"])
)

print("\nCompleted.")

=== HISTORICAL VOLATILITY SUMMARY ===


,Country,Name,Ticker,Observations,Valid_Returns,Valid_Volatility_5,Valid_Volatility_10
0,Malaysia,Affin Bank,5185.KL,2885,2883,2875,2865
1,Malaysia,Alliance Bank,2488.KL,2887,2885,2877,2867
2,Malaysia,Ambank,1015.KL,2892,2888,2872,2852
3,Malaysia,BIMB,5258.KL,2878,2869,2833,2788
4,Malaysia,CIMB,1023.KL,2896,2895,2891,2886
5,Malaysia,Hong Leong,5819.KL,2896,2895,2891,2886
6,Malaysia,MBSB,1171.KL,2895,2893,2885,2875
7,Malaysia,Maybank,1155.KL,2896,2895,2891,2886
8,Malaysia,Public Bank,1295.KL,2894,2892,2884,2874
9,Malaysia,RHB,1066.KL,2875,2871,2855,2835



=== SAMPLE VOLATILITY OBSERVATIONS ===


,Date,Close,returns,volatility_5,volatility_10
8663,2015-01-02,9.12,NaN,NaN,NaN
8664,2015-01-05,9.00,-0.013158,NaN,NaN
8665,2015-01-06,8.80,-0.022222,NaN,NaN
8666,2015-01-07,8.61,-0.021591,NaN,NaN
8667,2015-01-08,8.70,0.010453,NaN,NaN
8668,2015-01-09,8.61,-0.010345,0.013254,NaN
8669,2015-01-12,8.58,-0.003484,0.013629,NaN
8670,2015-01-13,8.62,0.004662,0.012588,NaN
8671,2015-01-14,8.39,-0.026682,0.014432,NaN
8672,2015-01-15,8.60,0.025030,0.019078,NaN



=== VOLATILITY DISTRIBUTION ===


volatility_5                     volatility_10                    
                  mean    median       max          mean    median       max
Country                                                                     
Malaysia      0.010975  0.009185  0.102728      0.011564  0.010043  0.078166
Singapore     0.009433  0.008163  0.065269      0.009957  0.008879  0.051295


Completed.


**Calculate price-based technical indicators**

In [4]:
# Group observations by stock and uninterrupted segment
groups = df.groupby(["Ticker", "Segment_ID"], sort=False)

# Calculate lagged daily returns
for lag in [1, 2]:
    df[f"returns_lag_{lag}"] = (
        groups["returns"].shift(lag)
    )

# Calculate historical price momentum
for window in [5, 10]:
    df[f"momentum_{window}"] = (
        df["Close"]
        / groups["Close"].shift(window)
        - 1
    )

# Calculate intraday price indicators
df["daily_range"] = (
    (df["High"] - df["Low"]) / df["Close"]
)

df["candle_body"] = (
    (df["Close"] - df["Open"]) / df["Open"]
)

# Calculate simple moving average ratios
for window in [5, 10, 20]:
    sma = groups["Close"].transform(
        lambda s: s.rolling(
            window=window,
            min_periods=window
        ).mean()
    )

    df[f"sma_{window}_ratio"] = (
        df["Close"] / sma - 1
    )

# Calculate 20-day historical volatility
df["volatility_20"] = (
    groups["returns"].transform(
        lambda s: s.rolling(
            window=20,
            min_periods=20
        ).std(ddof=1)
    )
)

# Define newly calculated features
price_features = [
    "returns_lag_1",
    "returns_lag_2",
    "momentum_5",
    "momentum_10",
    "daily_range",
    "candle_body",
    "sma_5_ratio",
    "sma_10_ratio",
    "sma_20_ratio",
    "volatility_20"
]

# Validate feature values
assert not np.isinf(
    df[price_features].to_numpy()
).any()

assert (df["daily_range"] >= 0).all()

# Summarise feature availability
feature_summary = pd.DataFrame({
    "Feature": price_features,
    "Valid_Observations": [
        df[column].notna().sum()
        for column in price_features
    ],
    "Missing_Observations": [
        df[column].isna().sum()
        for column in price_features
    ]
})

print("=== PRICE FEATURE SUMMARY ===")
display(feature_summary)

print("\n=== SAMPLE MAYBANK FEATURES ===")

display(
    df.loc[
        df["Ticker"] == "1155.KL",
        [
            "Date",
            "Close",
            "returns_lag_1",
            "momentum_5",
            "daily_range",
            "sma_5_ratio",
            "volatility_20"
        ]
    ].head(25)
)

print("\nCompleted.")

=== PRICE FEATURE SUMMARY ===


,Feature,Valid_Observations,Missing_Observations
0,returns_lag_1,37736,62
1,returns_lag_2,37705,93
2,momentum_5,37643,155
3,momentum_10,37488,310
4,daily_range,37798,0
5,candle_body,37798,0
6,sma_5_ratio,37674,124
7,sma_10_ratio,37519,279
8,sma_20_ratio,37211,587
9,volatility_20,37181,617



=== SAMPLE MAYBANK FEATURES ===


,Date,Close,returns_lag_1,momentum_5,daily_range,sma_5_ratio,volatility_20
8663,2015-01-02,9.12,NaN,NaN,0.013158,NaN,NaN
8664,2015-01-05,9.00,NaN,NaN,0.021111,NaN,NaN
8665,2015-01-06,8.80,-0.013158,NaN,0.040909,NaN,NaN
8666,2015-01-07,8.61,-0.022222,NaN,0.013937,NaN,NaN
8667,2015-01-08,8.70,-0.021591,NaN,0.013793,-0.016505,NaN
8668,2015-01-09,8.61,0.010453,-0.055921,0.016260,-0.015325,NaN
8669,2015-01-12,8.58,-0.010345,-0.046667,0.017483,-0.009238,NaN
8670,2015-01-13,8.62,-0.003484,-0.020455,0.011601,-0.000464,NaN
8671,2015-01-14,8.39,0.004662,-0.025552,0.028605,-0.022145,NaN
8672,2015-01-15,8.60,-0.026682,-0.011494,0.025581,0.004673,NaN



Completed.


**Calculate RSI, MACD and volume indicators**

In [5]:
# Calculate RSI using Wilder's smoothing method
def calculate_rsi(close, period=14):
    changes = close.diff().to_numpy(dtype=float)

    gains = np.maximum(changes, 0)
    losses = np.maximum(-changes, 0)

    rsi = np.full(len(close), np.nan)

    if len(close) <= period:
        return pd.Series(rsi, index=close.index)

    avg_gain = gains[1:period + 1].mean()
    avg_loss = losses[1:period + 1].mean()

    for i in range(period, len(close)):
        if i > period:
            avg_gain = (
                avg_gain * (period - 1) + gains[i]
            ) / period

            avg_loss = (
                avg_loss * (period - 1) + losses[i]
            ) / period

        if avg_gain == 0 and avg_loss == 0:
            rsi[i] = 50.0
        elif avg_loss == 0:
            rsi[i] = 100.0
        else:
            rs = avg_gain / avg_loss
            rsi[i] = 100 - 100 / (1 + rs)

    return pd.Series(rsi, index=close.index)


df["rsi"] = (
    df.groupby(["Ticker", "Segment_ID"])["Close"]
    .transform(calculate_rsi)
)

# Calculate 12-day and 26-day exponential moving averages
ema_12 = groups["Close"].transform(
    lambda s: s.ewm(
        span=12,
        adjust=False,
        min_periods=12
    ).mean()
)

ema_26 = groups["Close"].transform(
    lambda s: s.ewm(
        span=26,
        adjust=False,
        min_periods=26
    ).mean()
)

# Calculate MACD and its 9-day signal line
macd_raw = ema_12 - ema_26

macd_signal_raw = (
    macd_raw.groupby(
        [df["Ticker"], df["Segment_ID"]]
    )
    .transform(
        lambda s: s.ewm(
            span=9,
            adjust=False,
            min_periods=9
        ).mean()
    )
)

# Normalise MACD indicators by closing price
df["macd"] = macd_raw / df["Close"]
df["macd_signal"] = macd_signal_raw / df["Close"]

df["macd_diff"] = (
    macd_raw - macd_signal_raw
) / df["Close"]

# Calculate daily volume change
df["volume_change"] = (
    groups["Volume"]
    .pct_change(fill_method=None)
)

# Calculate volume relative to its 20-day average
volume_ma_20 = groups["Volume"].transform(
    lambda s: s.rolling(
        window=20,
        min_periods=20
    ).mean()
)

df["volume_ratio_20"] = (
    df["Volume"] / volume_ma_20 - 1
)

# Define newly calculated indicators
indicator_features = [
    "rsi",
    "macd",
    "macd_signal",
    "macd_diff",
    "volume_change",
    "volume_ratio_20"
]

# Validate indicator values
assert df["rsi"].dropna().between(0, 100).all()

assert not np.isinf(
    df[indicator_features].to_numpy()
).any()

# Summarise feature availability
indicator_summary = pd.DataFrame({
    "Feature": indicator_features,
    "Valid_Observations": [
        df[column].notna().sum()
        for column in indicator_features
    ],
    "Missing_Observations": [
        df[column].isna().sum()
        for column in indicator_features
    ]
})

print("=== TECHNICAL INDICATOR SUMMARY ===")
display(indicator_summary)

print("\n=== SAMPLE MAYBANK INDICATORS ===")

display(
    df.loc[
        (df["Ticker"] == "1155.KL")
        & df["Date"].between("2015-02-02", "2015-02-20"),
        [
            "Date",
            "Close",
            "rsi",
            "macd",
            "macd_signal",
            "macd_diff",
            "volume_change",
            "volume_ratio_20"
        ]
    ]
)

print("\n=== RSI DISTRIBUTION BY MARKET ===")

display(
    df.groupby("Country")["rsi"]
    .agg(["count", "mean", "median", "min", "max"])
)

print("\nCompleted.")

=== TECHNICAL INDICATOR SUMMARY ===


,Feature,Valid_Observations,Missing_Observations
0,rsi,37364,434
1,macd,37031,767
2,macd_signal,36793,1005
3,macd_diff,36793,1005
4,volume_change,37767,31
5,volume_ratio_20,37211,587



=== SAMPLE MAYBANK INDICATORS ===


,Date,Close,rsi,macd,macd_signal,macd_diff,volume_change,volume_ratio_20
8684,2015-02-04,8.83,44.305459,NaN,NaN,NaN,0.181018,0.012451
8685,2015-02-05,8.92,47.797602,NaN,NaN,NaN,-0.165256,-0.128806
8686,2015-02-06,8.93,48.186348,NaN,NaN,NaN,-0.152729,-0.217469
8687,2015-02-09,9.00,50.940460,NaN,NaN,NaN,-0.337377,-0.467421
8688,2015-02-10,9.10,54.649062,0.001639,NaN,NaN,0.312861,-0.262198
8689,2015-02-11,9.10,54.649062,0.003327,NaN,NaN,-0.023877,-0.274920
8690,2015-02-12,9.07,53.248555,0.004364,NaN,NaN,0.237958,-0.114526
8691,2015-02-13,9.19,57.896543,0.006092,NaN,NaN,-0.416251,-0.439661
8692,2015-02-16,9.20,58.268877,0.007500,NaN,NaN,-0.400074,-0.642375
8693,2015-02-17,9.20,58.268877,0.008523,NaN,NaN,-0.362742,-0.760502



=== RSI DISTRIBUTION BY MARKET ===


,count,mean,median,min,max
Country,,,,,
Malaysia,28502,50.118466,49.988695,8.266977,92.628584
Singapore,8862,52.844315,53.010532,8.851562,92.160421



Completed.


**Calculate Bollinger Bands and return skewness**

In [6]:
# Calculate 20-day closing price statistics
bb_middle = groups["Close"].transform(
    lambda s: s.rolling(
        window=20,
        min_periods=20
    ).mean()
)

bb_std = groups["Close"].transform(
    lambda s: s.rolling(
        window=20,
        min_periods=20
    ).std(ddof=1)
)

# Calculate upper and lower Bollinger Bands
bb_upper = bb_middle + 2 * bb_std
bb_lower = bb_middle - 2 * bb_std

# Normalise Bollinger Band values
df["bb_high"] = bb_upper / df["Close"] - 1
df["bb_low"] = bb_lower / df["Close"] - 1

# Calculate relative Bollinger Band width
df["bb_width"] = (
    (bb_upper - bb_lower) / bb_middle
)

# Calculate 20-day return skewness
df["skewness_20"] = groups["returns"].transform(
    lambda s: s.rolling(
        window=20,
        min_periods=20
    ).skew()
)

# Define newly calculated indicators
distribution_features = [
    "bb_high",
    "bb_low",
    "bb_width",
    "skewness_20"
]

# Validate calculated indicators
assert not np.isinf(
    df[distribution_features].to_numpy()
).any()

assert (df["bb_width"].dropna() >= 0).all()

# Summarise feature availability
distribution_summary = pd.DataFrame({
    "Feature": distribution_features,
    "Valid_Observations": [
        df[column].notna().sum()
        for column in distribution_features
    ],
    "Missing_Observations": [
        df[column].isna().sum()
        for column in distribution_features
    ]
})

print("=== DISTRIBUTION INDICATOR SUMMARY ===")
display(distribution_summary)

print("\n=== SAMPLE MAYBANK INDICATORS ===")

display(
    df.loc[
        (df["Ticker"] == "1155.KL")
        & df["Date"].between("2015-02-02", "2015-02-20"),
        [
            "Date",
            "Close",
            "bb_high",
            "bb_low",
            "bb_width",
            "skewness_20"
        ]
    ]
)

print("\n=== INDICATOR DISTRIBUTION BY MARKET ===")

display(
    df.groupby("Country")[
        ["bb_width", "skewness_20"]
    ].agg(["count", "mean", "median"])
)

print("\nCompleted.")

=== DISTRIBUTION INDICATOR SUMMARY ===


,Feature,Valid_Observations,Missing_Observations
0,bb_high,37211,587
1,bb_low,37211,587
2,bb_width,37211,587
3,skewness_20,37181,617



=== SAMPLE MAYBANK INDICATORS ===


,Date,Close,bb_high,bb_low,bb_width,skewness_20
8684,2015-02-04,8.83,0.024434,-0.045952,0.071152,-0.188735
8685,2015-02-05,8.92,0.015865,-0.056000,0.073337,-0.379516
8686,2015-02-06,8.93,0.017010,-0.055756,0.074204,-0.487602
8687,2015-02-09,9.00,0.012569,-0.063235,0.077774,-0.473191
8688,2015-02-10,9.10,0.006416,-0.073119,0.082279,-0.694913
8689,2015-02-11,9.10,0.010274,-0.071263,0.084102,-0.742824
8690,2015-02-12,9.07,0.016412,-0.066026,0.084535,-0.638997
8691,2015-02-13,9.19,0.003633,-0.070009,0.076170,-0.574333
8692,2015-02-16,9.20,0.005803,-0.067759,0.075914,-0.791700
8693,2015-02-17,9.20,0.007826,-0.063260,0.073113,-0.791700



=== INDICATOR DISTRIBUTION BY MARKET ===


bb_width                     skewness_20                    
             count      mean    median       count      mean    median
Country                                                               
Malaysia     28364  0.076701  0.062682       28337  0.086758  0.075309
Singapore     8847  0.078682  0.065491        8844  0.028162  0.047332


Completed.


**Consolidate and validate engineered features**

In [7]:
# Cell 5: Consolidate and validate engineered features

# Define candidate model features
feature_columns = [
    "returns",
    "returns_lag_1",
    "returns_lag_2",
    "momentum_5",
    "momentum_10",
    "daily_range",
    "candle_body",
    "sma_5_ratio",
    "sma_10_ratio",
    "sma_20_ratio",
    "volatility_5",
    "volatility_10",
    "volatility_20",
    "rsi",
    "macd",
    "macd_signal",
    "macd_diff",
    "volume_change",
    "volume_ratio_20",
    "bb_high",
    "bb_low",
    "bb_width",
    "skewness_20"
]

# Validate feature columns
assert all(
    column in df.columns
    for column in feature_columns
)

assert not np.isinf(
    df[feature_columns].to_numpy()
).any()

assert not df.duplicated(["Ticker", "Date"]).any()

# Identify observations with available historical volatility
df["Eligible_Volatility_5"] = (
    df["volatility_5"].notna()
    & (df["volatility_5"] > 0)
)

df["Eligible_Volatility_10"] = (
    df["volatility_10"].notna()
    & (df["volatility_10"] > 0)
)

# Identify observations with complete candidate model features
df["Eligible_All_Features"] = (
    df[feature_columns].notna().all(axis=1)
)

# Summarise feature availability by stock
eligibility_summary = (
    df.groupby(["Country", "Name", "Ticker"])
    .agg(
        Total_Rows=("Date", "size"),
        Eligible_Volatility_5=("Eligible_Volatility_5", "sum"),
        Eligible_Volatility_10=("Eligible_Volatility_10", "sum"),
        Eligible_All_Features=("Eligible_All_Features", "sum")
    )
    .reset_index()
)

print("=== FEATURE ENGINEERING SUMMARY ===")

print("Stocks:", df["Ticker"].nunique())
print("Observations:", len(df))
print("Candidate features:", len(feature_columns))

print("\n=== FEATURE ELIGIBILITY BY STOCK ===")
display(eligibility_summary)

print("\n=== OVERALL FEATURE AVAILABILITY ===")

availability_summary = pd.DataFrame({
    "Condition": [
        "5-day volatility available and positive",
        "10-day volatility available and positive",
        "All candidate model features available"
    ],
    "Observations": [
        int(df["Eligible_Volatility_5"].sum()),
        int(df["Eligible_Volatility_10"].sum()),
        int(df["Eligible_All_Features"].sum())
    ]
})

availability_summary["Percentage"] = (
    availability_summary["Observations"]
    / len(df) * 100
).round(2)

display(availability_summary)

# Review missing values in candidate features
missing_summary = (
    df[feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

missing_summary.columns = [
    "Feature",
    "Missing_Values"
]

print("\n=== MISSING FEATURE VALUES ===")
display(missing_summary)

# Preserve all historical observations and computed features
feature_data = df.copy()

# Save feature dataset
feature_path = (
    PROCESSED_DIR / "lseg_feature_engineered.csv"
)

feature_data.to_csv(
    feature_path,
    index=False
)

# Save feature eligibility summary
eligibility_summary.to_csv(
    RESULTS_DIR / "lseg_feature_eligibility.csv",
    index=False
)

print("\nFeature dataset saved:", feature_path)
print("Completed.")

=== FEATURE ENGINEERING SUMMARY ===
Stocks: 13
Observations: 37798
Candidate features: 23

=== FEATURE ELIGIBILITY BY STOCK ===


,Country,Name,Ticker,Total_Rows,Eligible_Volatility_5,Eligible_Volatility_10,Eligible_All_Features
0,Malaysia,Affin Bank,5185.KL,2885,2869,2865,2819
1,Malaysia,Alliance Bank,2488.KL,2887,2877,2867,2821
2,Malaysia,Ambank,1015.KL,2892,2872,2852,2760
3,Malaysia,BIMB,5258.KL,2878,2831,2788,2599
4,Malaysia,CIMB,1023.KL,2896,2891,2886,2863
5,Malaysia,Hong Leong,5819.KL,2896,2891,2886,2863
6,Malaysia,MBSB,1171.KL,2895,2877,2875,2829
7,Malaysia,Maybank,1155.KL,2896,2891,2886,2863
8,Malaysia,Public Bank,1295.KL,2894,2878,2874,2828
9,Malaysia,RHB,1066.KL,2875,2855,2835,2743



=== OVERALL FEATURE AVAILABILITY ===


,Condition,Observations,Percentage
0,5-day volatility available and positive,37621,99.53
1,10-day volatility available and positive,37488,99.18
2,All candidate model features available,36793,97.34



=== MISSING FEATURE VALUES ===


,Feature,Missing_Values
0,macd_diff,1005
1,macd_signal,1005
2,macd,767
3,volatility_20,617
4,skewness_20,617
5,volume_ratio_20,587
6,sma_20_ratio,587
7,bb_width,587
8,bb_high,587
9,bb_low,587



Feature dataset saved: C:\UTAR - Master\Master Project\Main Project\data\processed\lseg_feature_engineered.csv
Completed.
